
# Importações

In [12]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error as mape, root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

from pyESN import ESN
from shap.plots import colors
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from wsb import WSB

MODELOS = ["ESN", "MLP", "RF", "XGBoost", "WSB"]
SEED = 100
SEEDS = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
HORIZONTES = [3, 6, 12]


def reset_seed(rnd_seed=SEED):
    os.environ['PYTHONHASHSEED'] = '0'
    random.seed(rnd_seed)
    np.random.seed(rnd_seed)


def calcular_rrmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)

    mean_y_true = np.mean(y_true)

    rrmse = rmse / mean_y_true
    return rrmse


warnings.filterwarnings("ignore")
reset_seed()

# Carregar Datasets

In [13]:
df = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

## Normalização

In [14]:
# Repete a normalização para obtermos so scalers correspondentes
scalers = {}
dataframes = []

for campus, dados in df.groupby("CAMPUS"):
    scaler = MinMaxScaler()
    dados[["CONSUMO"]] = scaler.fit_transform(dados[["CONSUMO"]])

    scalers[campus] = scaler
    dataframes.append(dados)

df = pd.concat(dataframes, ignore_index=True)

## Criação dos Lags

In [15]:
dataframes = []

for campus, dados in df.sort_values("DATA").groupby("CAMPUS"):
    lags = {f'LAG_{i:02d}': dados['CONSUMO'].shift(i) for i in range(1, 12 + 1)}
    dados = pd.concat([dados, pd.DataFrame(lags)], axis=1)
    dados.dropna(inplace=True)
    dados["ORDEM"] = range(1, len(dados) + 1)
    dataframes.append(dados)

df = pd.concat(dataframes, ignore_index=True)
df

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,LAG_03,LAG_04,LAG_05,LAG_06,LAG_07,LAG_08,LAG_09,LAG_10,LAG_11,LAG_12
0,0.615105,2016-02-29,19,22,25,28,734,33,4,19,...,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886,0.502037
1,0.711047,2016-03-31,12,17,22,28,685,32,3,12,...,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886
2,0.633361,2016-04-30,4,9,23,28,702,33,2,4,...,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078
3,0.406291,2016-05-31,4,11,16,22,490,27,9,4,...,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844
4,0.362467,2016-06-30,-1,7,14,20,408,27,3,-1,...,0.711047,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2090,0.359224,2024-06-30,0,8,17,20,499,26,1,0,...,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248,0.743880
2091,1.000000,2024-07-31,1,9,13,18,413,25,6,1,...,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248
2092,0.940871,2024-08-31,-3,7,15,21,468,31,1,-3,...,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038
2093,0.788386,2024-09-30,10,13,20,24,591,34,3,10,...,0.359224,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059


## Melhores Features

In [16]:
df_features = pd.read_csv("./resultados/features/fitness_features_regressao.csv", sep=";", decimal=".")

df_features = df_features.sort_values("RRMSE").head(1).reset_index(drop=True)
df_features = pd.DataFrame(
    columns=str(df_features.iloc[0]["FEATURES"]).replace("(", '').replace(")", '').replace("'", "").split(", "))

df_features = df_features.columns

df_features

Index(['TEMP_MÉD_MIN_MENS', 'TEMP_MÉD_MÉD_MENS', 'PRECIPITAÇÃO_MÉD_MENS',
       'TEMP_MIN_MAX_MENS', 'TEMP_MAX_MIN_MENS', 'PRECIPITAÇÃO_MIN_MENS',
       'TEMP_MAX_MAX_MENS', 'DIA_DA_SEMANA_dom', 'DIA_DA_SEMANA_seg',
       'DIA_DA_SEMANA_sáb', 'DIA_DA_SEMANA_ter', 'MÊS_abr', 'MÊS_ago',
       'MÊS_fev', 'MÊS_jun', 'MÊS_mai', 'MÊS_nov', 'ANO_2021', 'ANO_2022',
       'ANO_2023', 'ANO_2015', 'ANO_2016', 'ANO_2017', 'ANO_2018', 'ANO_2019',
       'CAMPUS_ASTORGA', 'CAMPUS_CAMPO LARGO', 'CAMPUS_CAPANEMA',
       'CAMPUS_CASCAVEL', 'CAMPUS_CORONEL VIVIDA', 'CAMPUS_CURITIBA',
       'CAMPUS_GOIOERÊ', 'CAMPUS_IVAIPORÃ', 'CAMPUS_JAGUARIAÍVA',
       'CAMPUS_LONDRINA - CENTRO', 'CAMPUS_PALMAS', 'CAMPUS_PARANAGUÁ',
       'CAMPUS_PINHAIS', 'CAMPUS_TELÊMACO BORBA', 'CAMPUS_UMUARAMA',
       'CURSOS_TEC_SUBSEQUENTE', 'CURSOS_GRAD_MATUTINO',
       'CURSOS_GRAD_VESPERTINO', 'CURSOS_GRAD_NOTURNO', 'CURSOS_POS', 'FÉRIAS',
       'COVID', 'LAG_01', 'LAG_02', 'LAG_03', 'LAG_05', 'LAG_07', 'LAG_09'],


## Melhores Parâmetros

In [17]:
def get_modelo(nome, tipo_treino=None, campus=None):
    if nome == "ESN":
        return ESN(n_inputs=df_features.shape[0],
                   n_outputs=1,
                   n_reservoir=int(best["ESN"]["Reservoirs"]),
                   sparsity=best["ESN"]["Sparsity"],
                   spectral_radius=best["ESN"]["Spectral Radius"],
                   random_state=int(best["ESN"]["SEED"]))

    if nome == "MLP":
        mlp = MLPRegressor(hidden_layer_sizes=(int(best["MLP"]["Hidden Layers"]),),
                           activation=best["MLP"]["Activation"],
                           alpha=best["MLP"]["Alpha"],
                           random_state=int(best["MLP"]["SEED"]))
        return mlp

    if nome == "RF":
        return RandomForestRegressor(random_state=int(best["RF"]["SEED"]),
                                     n_estimators=int(best["RF"]["N_estimators"]),
                                     max_depth=int(best["RF"]["Max_depth"]),
                                     min_samples_split=int(best["RF"]["Min_samples_split"]),
                                     min_samples_leaf=int(best["RF"]["Min_samples_leaf"]))

    if nome == "XGBoost":
        return XGBRegressor(random_state=int(best["XGBoost"]["SEED"]),
                            n_estimators=int(best["XGBoost"]["N_estimators"]),
                            max_depth=int(best["XGBoost"]["Max_depth"]),
                            booster=best["XGBoost"]["Booster"],
                            reg_lambda=best["XGBoost"]["Lambda"],
                            reg_alpha=best["XGBoost"]["Alpha"],
                            updater="coord_descent" if best["XGBoost"]["Booster"] == "gblinear" else None)

    if nome == "WSB":
        return WSB(strong_predictor=get_modelo(wsb_prev_forte[tipo_treino]),
                   weak_predictors=[get_modelo(m) for m in MODELOS if m != "WSB" and m != strong_pred],
                   weight_g=wsb_pesos[(campus, tipo_treino)])


wsb_prev_forte = {"local": "XGBoost", "global": "ESN"}
wsb_pesos = {}
best = {}
for modelo in MODELOS:
    if modelo == "WSB":
        continue
    df_aux = pd.read_csv(
        f"./resultados/otimização - regressão/BEST {modelo}.csv", sep=';',
        decimal='.', header=0)

    best[modelo] = df_aux.iloc[0]

for key, val in best.items():
    display(val)

OTIMIZADOR           PSO
MODELO               ESN
SEED                9000
Reservoirs          11.0
Sparsity            0.23
Spectral Radius    0.639
Fitness            0.289
Name: 0, dtype: object

OTIMIZADOR         PSO
MODELO             MLP
SEED             10000
Hidden Layers      220
Alpha            0.979
Activation        relu
Fitness          0.227
Name: 0, dtype: object

OTIMIZADOR             PSO
MODELO                  RF
SEED                  7000
N_estimators          15.0
Max_depth            232.0
Min_samples_split     11.0
Min_samples_leaf       4.0
Fitness              0.247
Name: 0, dtype: object

OTIMIZADOR          PSO
MODELO          XGBoost
SEED               5000
N_estimators        242
Max_depth           119
Booster          gbtree
Lambda            0.898
Alpha             0.041
Fitness           0.243
Name: 0, dtype: object

# Divisão dos Dados


In [18]:
dfs_treino = {}
dfs_teste = {}
for horizonte in HORIZONTES:
    treino = []
    teste = []

    for campus, dados in df.sort_values('DATA').groupby("CAMPUS"):
        dados["CAMPUS"] = campus

        dados_treino, dados_teste = train_test_split(dados, test_size=horizonte, shuffle=False)

        treino.append(dados_treino)
        teste.append(dados_teste)

    treino = pd.DataFrame(pd.concat(treino, ignore_index=True))
    teste = pd.DataFrame(pd.concat(teste, ignore_index=True))

    dfs_treino[horizonte] = treino
    dfs_teste[horizonte] = teste



# Execução dos Experimentos

In [19]:
def treino(previsor, dados_treino, features):
    x_treino = dados_treino[features].to_numpy()
    y_treino = dados_treino["CONSUMO"].to_numpy()

    previsor.fit(x_treino, y_treino)
    return previsor


def teste(previsor, historico_campus, x_teste, features, horizonte):
    historico = historico_campus[["CONSUMO"]].copy()
    x_teste = x_teste[features].copy()

    previsoes = []

    for i_test in range(horizonte):
        row = x_teste.iloc[[i_test]].copy()
        historico = pd.concat([historico, pd.DataFrame([0], columns=["CONSUMO"], index=[i_test])], axis=0)

        # Recalcula os lags conforme os valores previstos pelo modelo
        lags = pd.DataFrame({f'LAG_{i:02d}': historico["CONSUMO"].shift(i) for i in range(1, 12 + 1) if
                             f'LAG_{i:02d}' in features}).tail(1)
        row.update(lags)

        if isinstance(previsor, WSB) and horizonte >= 6:
            peso_t = i_test / horizonte
            prev = previsor.predict(row.to_numpy(), peso_t)[0]
        elif isinstance(previsor, ESN):
            prev = previsor.predict(row.to_numpy())[0][0]
        else:
            prev = previsor.predict(row.to_numpy())[0]

        row["CONSUMO"] = prev
        previsoes.append(prev)
        historico.update(row)

    return pd.DataFrame({"CONSUMO PREVISTO": previsoes}, index=x_teste.index)

## Definição dos Previsores Fortes - Local

In [20]:
# Treinamento local
for horizonte in HORIZONTES:
    for campus, dados in dfs_treino[horizonte].sort_values("DATA").groupby("CAMPUS"):
        dados = dados.set_index('DATA')
        dados_treino, dados_teste = train_test_split(dados, test_size=horizonte, shuffle=False)

        erros = []
        # Treina o modelo com os dados do campus atual
        strong_pred = get_modelo(wsb_prev_forte["local"])
        weak_preds = [get_modelo(m) for m in MODELOS if m != "WSB" and m != strong_pred]
        modelo = WSB(strong_predictor=strong_pred, weak_predictors=weak_preds, weight_g=-1)
        modelo = treino(modelo, dados, df_features)

        # Testa o modelo com os dados do campus atual
        previsoes = teste(modelo, dados, dados_teste, df_features, horizonte)
        erro = (dados_teste["CONSUMO"].mean() - previsoes["CONSUMO PREVISTO"].mean()) / dados_teste["CONSUMO"].mean()

        # Atualiza o peso individual do campus
        wsb_pesos[(campus, "local")] = -1 + erro

    # Converte o dicionário wsb_prev_forte em um DataFrame
    df_wsb_pesos = pd.DataFrame.from_dict(wsb_pesos, orient="index", columns=["WEIGHT_G"])
    df_wsb_pesos.index = pd.MultiIndex.from_tuples(df_wsb_pesos.index, names=["CAMPUS", "TIPO_TREINAMENTO"])

    # Salva o DataFrame em um arquivo CSV
    df_wsb_pesos.to_csv(f"./resultados/WSB - Previsores Fortes {horizonte}M.csv", sep=';', decimal='.')
    display(df_wsb_pesos)

,,WEIGHT_G
CAMPUS,TIPO_TREINAMENTO,
ASSIS CHATEAUBRIAND,local,-1.191047
ASTORGA,local,-0.987997
BARRACÃO,local,-1.016882
CAMPO LARGO,local,-1.531455
CAPANEMA,local,-1.201758
CASCAVEL,local,-1.559937
CORONEL VIVIDA,local,-0.945570
CURITIBA,local,-1.735160
FOZ DO IGUAÇU,local,-1.222505


,,WEIGHT_G
CAMPUS,TIPO_TREINAMENTO,
ASSIS CHATEAUBRIAND,local,-0.980699
ASTORGA,local,-0.980289
BARRACÃO,local,-0.962277
CAMPO LARGO,local,-1.009235
CAPANEMA,local,-0.995431
CASCAVEL,local,-0.980343
CORONEL VIVIDA,local,-1.042649
CURITIBA,local,-0.894920
FOZ DO IGUAÇU,local,-1.048677


,,WEIGHT_G
CAMPUS,TIPO_TREINAMENTO,
ASSIS CHATEAUBRIAND,local,-0.997149
ASTORGA,local,-0.993090
BARRACÃO,local,-0.974075
CAMPO LARGO,local,-1.007456
CAPANEMA,local,-1.009112
CASCAVEL,local,-0.985850
CORONEL VIVIDA,local,-0.991439
CURITIBA,local,-1.012236
FOZ DO IGUAÇU,local,-1.013342


## Definição dos Previsores Fortes - Global

In [21]:
# Treinamento global
for horizonte in HORIZONTES:
    df_aux_treino = []
    df_aux_test = []

    for campus, dados in dfs_treino[horizonte].sort_values("DATA").groupby("CAMPUS"):
        dados_treino, dados_teste = train_test_split(dados, test_size=horizonte, shuffle=False)
        df_aux_treino.append(dados_treino)
        df_aux_test.append(dados_teste)

    df_aux_treino = pd.DataFrame(pd.concat(df_aux_treino, ignore_index=True))
    df_aux_test = pd.DataFrame(pd.concat(df_aux_test, ignore_index=True))

    for campus, dados in df_aux_test.sort_values("DATA").groupby("CAMPUS"):
        dados = dados.set_index('DATA')

        # Treina o modelo com os dados de todos os campi
        strong_pred = get_modelo(wsb_prev_forte["global"])
        weak_preds = [get_modelo(m) for m in MODELOS if m != "WSB" and m != strong_pred]
        modelo = WSB(strong_predictor=strong_pred, weak_predictors=weak_preds, weight_g=-1)
        modelo = treino(modelo, df_aux_treino, df_features)

        # Testa o modelo com os dados do campus atual
        previsoes = teste(modelo, df_aux_treino[df_aux_treino["CAMPUS"] == campus], dados, df_features, horizonte)
        erro = (dados["CONSUMO"].mean() - previsoes["CONSUMO PREVISTO"].mean()) / dados["CONSUMO"].mean()

        # Atualiza o peso individual do campus
        wsb_pesos[(campus, "global")] = -1 + erro

    # Converte o dicionário wsb_prev_forte em um DataFrame
    df_wsb_pesos = pd.DataFrame.from_dict(wsb_pesos, orient="index", columns=["WEIGHT_G"])
    df_wsb_pesos.index = pd.MultiIndex.from_tuples(df_wsb_pesos.index, names=["CAMPUS", "TIPO_TREINAMENTO"])

    # Salva o DataFrame em um arquivo CSV
    df_wsb_pesos.to_csv(f"./resultados/WSB - Previsores Fortes {horizonte}M.csv", sep=';', decimal='.')
    display(df_wsb_pesos)

,,WEIGHT_G
CAMPUS,TIPO_TREINAMENTO,
ASSIS CHATEAUBRIAND,local,-0.997149
ASTORGA,local,-0.993090
BARRACÃO,local,-0.974075
CAMPO LARGO,local,-1.007456
CAPANEMA,local,-1.009112
CASCAVEL,local,-0.985850
CORONEL VIVIDA,local,-0.991439
CURITIBA,local,-1.012236
FOZ DO IGUAÇU,local,-1.013342


,,WEIGHT_G
CAMPUS,TIPO_TREINAMENTO,
ASSIS CHATEAUBRIAND,local,-0.997149
ASTORGA,local,-0.993090
BARRACÃO,local,-0.974075
CAMPO LARGO,local,-1.007456
CAPANEMA,local,-1.009112
CASCAVEL,local,-0.985850
CORONEL VIVIDA,local,-0.991439
CURITIBA,local,-1.012236
FOZ DO IGUAÇU,local,-1.013342


,,WEIGHT_G
CAMPUS,TIPO_TREINAMENTO,
ASSIS CHATEAUBRIAND,local,-0.997149
ASTORGA,local,-0.993090
BARRACÃO,local,-0.974075
CAMPO LARGO,local,-1.007456
CAPANEMA,local,-1.009112
CASCAVEL,local,-0.985850
CORONEL VIVIDA,local,-0.991439
CURITIBA,local,-1.012236
FOZ DO IGUAÇU,local,-1.013342




## Treinamento Local

In [22]:
for horizonte in HORIZONTES:
    df_treino = dfs_treino[horizonte]
    for campus, dados_teste in dfs_teste[horizonte].sort_values("DATA").groupby("CAMPUS"):
        dados_teste = dados_teste.set_index('DATA')
        df_previsoes = pd.DataFrame(columns=MODELOS, index=dados_teste.index)

        for nome_modelo in MODELOS:
            # Treina o modelo com os dados do campus atual
            dados_treino = df_treino[df_treino["CAMPUS"] == campus]
            modelo = treino(get_modelo(nome_modelo, tipo_treino="local", campus=campus), dados_treino, df_features)

            # Testa o modelo com os dados do campus atual
            previsoes = teste(modelo, dados_treino, dados_teste, df_features, horizonte)
            df_previsoes[[nome_modelo]] = scalers[campus].inverse_transform(previsoes[["CONSUMO PREVISTO"]])

        os.makedirs(f"resultados/regressão - local/{horizonte} meses", exist_ok=True)
        df_previsoes.to_csv(f"resultados/regressão - local/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                            sep=";", decimal=".", header=True,
                            index=True)


## Treinamento Global

In [23]:

for horizonte in HORIZONTES:
    df_treino = dfs_treino[horizonte]
    for campus, dados_teste in dfs_teste[horizonte].sort_values("DATA").groupby("CAMPUS"):
        dados_teste = dados_teste.set_index('DATA')
        df_previsoes = pd.DataFrame(columns=MODELOS, index=dados_teste.index)

        for nome_modelo in MODELOS:
            # Treina o modelo com os dados de todos os campi
            modelo = treino(get_modelo(nome_modelo, tipo_treino="global", campus=campus), df_treino, df_features)

            # Testa o modelo com os dados do campus atual
            previsoes = teste(modelo, df_treino[df_treino["CAMPUS"] == campus], dados_teste, df_features, horizonte)
            df_previsoes[[nome_modelo]] = scalers[campus].inverse_transform(previsoes[["CONSUMO PREVISTO"]])

        os.makedirs(f"resultados/regressão - global/{horizonte} meses", exist_ok=True)
        df_previsoes.to_csv(f"resultados/regressão - global/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                            sep=";", decimal=".",
                            index=True)



# Análise dos Resultados

In [24]:
def ts_comparacao(campus, valor_real, valores_previstos, erros, horizonte):
    valor_real = valor_real.tail(horizonte)
    plt.figure(figsize=(12, 4.5))
    plt.rcParams['xtick.labelsize'] = 13
    plt.rcParams['ytick.labelsize'] = 14
    plt.rcParams.update({'font.size': 12})
    plt.rcParams['axes.prop_cycle'] = plt.cycler(
        color=["blue", "green", "darkgoldenrod", colors.red_rgb, "purple", "cyan", "slategrey", "coral"])

    for nome_modelo in valores_previstos.columns:
        plt.plot(valores_previstos[nome_modelo],
                 label=f"{nome_modelo} (RRMSE: {erros.loc[nome_modelo]["RRMSE"]:.2%} - MAPE: {erros.loc[nome_modelo]["MAPE"]:.2%})")

    plt.plot(valor_real["CONSUMO"], label=f"CONSUMO REAL - {campus}", color="black")

    plt.xlabel('Mês')
    plt.ylabel('Consumo (KWh)')

    ax = plt.gca()
    ax.set_facecolor('white')

    plt.grid(True, color='grey', linestyle="--", linewidth=0.5)
    plt.legend(facecolor='white')

    return plt


def medidas_desempenho(valor_real, valores_previstos, horizonte):
    df_desempenho = pd.DataFrame(columns=["MAPE", "RRMSE"], index=valores_previstos.columns)

    for nome_modelo in valores_previstos.columns:
        df_desempenho.loc[nome_modelo] = [
            mape(valor_real["CONSUMO"].tail(horizonte), valores_previstos[nome_modelo].tail(horizonte)),
            calcular_rrmse(valor_real["CONSUMO"].tail(horizonte), valores_previstos[nome_modelo].tail(horizonte)),
        ]

    return df_desempenho


## Treinamento Local

In [28]:
df_real = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

for horizonte in HORIZONTES:
    df_RRMSE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())
    df_MAPE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())

    for campus, dados in df_real.sort_values("DATA").groupby("CAMPUS"):
        try:
            consumo_previsto = pd.read_csv(
                f"resultados/regressão - local/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                sep=";", decimal=".", header=0)
        except Exception as e:
            continue

        consumo_previsto["DATA"] = pd.to_datetime(consumo_previsto["DATA"])
        consumo_previsto = consumo_previsto.set_index("DATA")

        dados["DATA"] = pd.to_datetime(dados["DATA"])
        dados = dados.set_index("DATA")

        df_desempenho = medidas_desempenho(dados, consumo_previsto, horizonte)
        df_RRMSE.loc[campus] = df_desempenho["RRMSE"]
        df_MAPE.loc[campus] = df_desempenho["MAPE"]

        plt = ts_comparacao(campus, dados, consumo_previsto, df_desempenho, horizonte)

        df_desempenho.to_csv(f"resultados/regressão - local/{horizonte} meses/RRMSE {horizonte}M {campus}.csv",
                             sep=";", decimal=".", index=True)
        plt.savefig(f"resultados/regressão - local/{horizonte} meses/PREVISÕES {horizonte}M {campus}.png",
                    bbox_inches='tight')
        plt.close()

    df_RRMSE = df_RRMSE.add_suffix(" RRMSE")
    df_MAPE = df_MAPE.add_suffix(" MAPE")

    pior_RRMSE = (df_RRMSE.eq(df_RRMSE.max(axis=1), axis=0).sum(axis=0))
    melhor_RRMSE = (df_RRMSE.eq(df_RRMSE.min(axis=1), axis=0).sum(axis=0))

    pior_MAPE = (df_MAPE.eq(df_MAPE.max(axis=1), axis=0).sum(axis=0))
    melhor_MAPE = (df_MAPE.eq(df_MAPE.min(axis=1), axis=0).sum(axis=0))

    df_medias = pd.concat([df_RRMSE, df_MAPE], axis=1)
    df_medias.loc["MÉDIAS"] = df_medias.mean()

    df_medias.loc["MELHOR"] = pd.concat([melhor_RRMSE, melhor_MAPE], axis=0)
    df_medias.loc["PIOR"] = pd.concat([pior_RRMSE, pior_MAPE], axis=0)
    df_medias.loc["SCORE (MELHOR - PIOR)"] = pd.concat([melhor_RRMSE - pior_RRMSE, melhor_MAPE - pior_MAPE], axis=0)

    df_medias.to_csv(f"resultados/regressão - local/MÉDIAS ERROS {horizonte}M.csv", sep=";", decimal=".", index=True)
    display(df_medias)



,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB MAPE
ASSIS CHATEAUBRIAND,0.311359,0.12425,0.296678,0.384678,0.28787,0.259414,0.064378,0.219504,0.289052,0.221084
ASTORGA,0.326683,0.278516,0.298948,0.334858,0.282408,0.272433,0.273608,0.27509,0.29229,0.257383
BARRACÃO,0.10952,0.159638,0.135852,0.272083,0.151288,0.096757,0.158585,0.135869,0.269534,0.151525
CAMPO LARGO,0.314383,0.286582,0.173849,0.306255,0.263246,0.275375,0.284823,0.169982,0.297039,0.249638
CAPANEMA,0.422843,0.203984,0.230291,0.166305,0.210543,0.315073,0.10328,0.171482,0.137143,0.128284
CASCAVEL,0.282968,0.178718,0.245105,0.265865,0.256957,0.253321,0.160683,0.177289,0.203063,0.198022
CORONEL VIVIDA,0.293099,0.239361,0.19786,0.218547,0.229331,0.22363,0.231081,0.16801,0.20223,0.213341
CURITIBA,0.962509,0.220101,0.167784,0.261969,0.085938,0.901734,0.133404,0.165749,0.245824,0.076106
FOZ DO IGUAÇU,0.222286,0.285499,0.118387,0.21132,0.205121,0.201441,0.247414,0.112985,0.203257,0.16135
GOIOERÊ,0.306091,0.18162,0.112599,0.281516,0.068387,0.289228,0.145535,0.099313,0.199324,0.064099


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB MAPE
ASSIS CHATEAUBRIAND,0.32826,0.52359,0.420644,0.42222,0.390806,0.25045,0.757365,0.511374,0.479986,0.469771
ASTORGA,0.307834,0.284013,0.288267,0.300552,0.265609,0.222226,0.267246,0.242436,0.222603,0.200764
BARRACÃO,0.250062,0.209844,0.198753,0.242066,0.214467,0.208184,0.209669,0.181716,0.221636,0.197388
CAMPO LARGO,0.467923,0.425668,0.363527,0.388361,0.369557,0.601344,0.542792,0.402035,0.459732,0.454286
CAPANEMA,0.508555,0.451082,0.401643,0.338276,0.339324,0.50463,0.623115,0.503296,0.412497,0.391999
CASCAVEL,0.47233,0.419661,0.365115,0.329403,0.319426,0.591357,0.466084,0.380758,0.382373,0.365445
CORONEL VIVIDA,0.245431,0.277802,0.217247,0.205905,0.194405,0.206578,0.282892,0.167997,0.152297,0.144124
CURITIBA,0.591001,0.342616,0.312017,0.288538,0.243009,0.29812,0.344478,0.338032,0.289919,0.259644
FOZ DO IGUAÇU,0.450966,0.4306,0.432367,0.338752,0.339184,0.421524,0.416376,0.43472,0.330228,0.319812
GOIOERÊ,0.469175,0.514052,0.350443,0.53874,0.517599,0.446817,0.542512,0.346408,0.477793,0.446678


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB MAPE
ASSIS CHATEAUBRIAND,0.280578,0.398155,0.36148,0.345205,0.344896,0.178217,0.551513,0.41961,0.41411,0.419487
ASTORGA,0.477217,0.308458,0.306776,0.334848,0.327267,0.430907,0.236664,0.228922,0.255418,0.245007
BARRACÃO,0.288687,0.229966,0.295104,0.27954,0.273175,0.233853,0.21752,0.26805,0.242562,0.240372
CAMPO LARGO,0.365603,0.334673,0.335989,0.326037,0.311579,0.358701,0.378977,0.355507,0.320603,0.328726
CAPANEMA,0.535447,0.431023,0.395956,0.340894,0.364174,0.470756,0.550737,0.436529,0.365215,0.396548
CASCAVEL,0.273705,0.298589,0.326426,0.283239,0.273274,0.283747,0.285663,0.328,0.256836,0.256368
CORONEL VIVIDA,0.345348,0.34397,0.267654,0.239236,0.254817,0.304481,0.379106,0.287886,0.250438,0.268652
CURITIBA,0.841661,0.393413,0.295968,0.354607,0.31337,0.668006,0.250875,0.264717,0.309176,0.260667
FOZ DO IGUAÇU,0.385475,0.512756,0.438348,0.485188,0.476748,0.423062,0.566678,0.510157,0.518693,0.52358
GOIOERÊ,0.25546,0.266759,0.304176,0.277809,0.269538,0.272904,0.298071,0.276366,0.281362,0.262461


## Treinamento Global

In [27]:
df_real = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

for horizonte in HORIZONTES:

    df_RRMSE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())
    df_MAPE = pd.DataFrame(columns=MODELOS, index=df_real["CAMPUS"].unique())

    for campus, dados in df_real.sort_values("DATA").groupby("CAMPUS"):
        try:
            consumo_previsto = pd.read_csv(
                f"resultados/regressão - global/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                sep=";", decimal=".", header=0)
        except Exception as e:
            continue

        consumo_previsto["DATA"] = pd.to_datetime(consumo_previsto["DATA"])
        consumo_previsto = consumo_previsto.set_index("DATA")

        dados["DATA"] = pd.to_datetime(dados["DATA"])
        dados = dados.set_index("DATA")

        df_desempenho = medidas_desempenho(dados, consumo_previsto, horizonte)
        df_RRMSE.loc[campus] = df_desempenho["RRMSE"]
        df_MAPE.loc[campus] = df_desempenho["MAPE"]

        plt = ts_comparacao(campus, dados, consumo_previsto, df_desempenho, horizonte)

        df_desempenho.to_csv(f"resultados/regressão - global/{horizonte} meses/RRMSE {horizonte}M {campus}.csv",
                             sep=";", decimal=".", index=True)
        plt.savefig(f"resultados/regressão - global/{horizonte} meses/PREVISÕES {horizonte}M {campus}.png",
                    bbox_inches='tight')
        plt.close()

    df_RRMSE = df_RRMSE.add_suffix(" RRMSE")
    df_MAPE = df_MAPE.add_suffix(" MAPE")

    pior_RRMSE = (df_RRMSE.eq(df_RRMSE.max(axis=1), axis=0).sum(axis=0))
    melhor_RRMSE = (df_RRMSE.eq(df_RRMSE.min(axis=1), axis=0).sum(axis=0))

    pior_MAPE = (df_MAPE.eq(df_MAPE.max(axis=1), axis=0).sum(axis=0))
    melhor_MAPE = (df_MAPE.eq(df_MAPE.min(axis=1), axis=0).sum(axis=0))

    df_medias = pd.concat([df_RRMSE, df_MAPE], axis=1)
    df_medias.loc["MÉDIAS"] = df_medias.mean()

    df_medias.loc["MELHOR"] = pd.concat([melhor_RRMSE, melhor_MAPE], axis=0)
    df_medias.loc["PIOR"] = pd.concat([pior_RRMSE, pior_MAPE], axis=0)
    df_medias.loc["SCORE (MELHOR - PIOR)"] = pd.concat([melhor_RRMSE - pior_RRMSE, melhor_MAPE - pior_MAPE], axis=0)

    df_medias.to_csv(f"resultados/regressão - global/MÉDIAS ERROS {horizonte}M.csv", sep=";", decimal=".", index=True)
    display(df_medias)



,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB MAPE
ASSIS CHATEAUBRIAND,0.335021,0.52028,0.241681,0.310252,0.30808,0.320783,0.415469,0.204475,0.210689,0.232188
ASTORGA,0.239765,0.400109,0.183562,0.11608,0.211597,0.219098,0.388658,0.176063,0.093383,0.197268
BARRACÃO,0.143302,0.255247,0.150445,0.167824,0.158387,0.125761,0.247998,0.139153,0.154716,0.142712
CAMPO LARGO,0.214925,0.346374,0.247012,0.248021,0.260631,0.184126,0.342332,0.219828,0.244664,0.245355
CAPANEMA,0.344694,0.485142,0.291509,0.278738,0.309365,0.332013,0.382279,0.261259,0.177098,0.252029
CASCAVEL,0.233249,0.370364,0.213974,0.254905,0.241626,0.169461,0.343652,0.188028,0.232148,0.216771
CORONEL VIVIDA,0.201665,0.29845,0.203325,0.222105,0.214436,0.169635,0.27545,0.178974,0.195599,0.186989
CURITIBA,0.321913,0.30471,0.057306,0.115496,0.206465,0.305627,0.297917,0.053383,0.107602,0.17333
FOZ DO IGUAÇU,0.237318,0.252213,0.111688,0.089633,0.124756,0.225089,0.235992,0.102689,0.075077,0.123002
GOIOERÊ,0.465857,0.142725,0.185727,0.263042,0.231487,0.438196,0.14076,0.193288,0.262948,0.226122


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB MAPE
ASSIS CHATEAUBRIAND,0.455575,0.563616,0.531734,0.40938,0.408036,0.480224,0.772425,0.744592,0.541963,0.463381
ASTORGA,0.228303,0.192581,0.193867,0.127091,0.191994,0.188281,0.183534,0.168523,0.117226,0.161906
BARRACÃO,0.21964,0.201621,0.237855,0.179884,0.211702,0.182349,0.188269,0.194322,0.152587,0.175596
CAMPO LARGO,0.276241,0.347924,0.31567,0.359276,0.271879,0.319383,0.447986,0.40275,0.445261,0.322454
CAPANEMA,0.408605,0.516813,0.467003,0.453281,0.3929,0.489924,0.678008,0.647627,0.584114,0.495529
CASCAVEL,0.291019,0.35645,0.332701,0.378977,0.283268,0.295189,0.402957,0.405603,0.429562,0.304275
CORONEL VIVIDA,0.230479,0.214701,0.256904,0.223953,0.219789,0.179715,0.21883,0.204143,0.169478,0.171944
CURITIBA,0.287094,0.545697,0.211593,0.257673,0.24725,0.26334,0.632672,0.217248,0.281225,0.235444
FOZ DO IGUAÇU,0.256414,0.627122,0.348797,0.317584,0.246699,0.230079,0.632343,0.307016,0.259493,0.224994
GOIOERÊ,0.445886,0.654663,0.525382,0.536829,0.455773,0.356874,0.677914,0.507368,0.502124,0.386679


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB MAPE
ASSIS CHATEAUBRIAND,0.383148,0.506564,0.385431,0.342546,0.362573,0.368745,0.697377,0.495534,0.437677,0.41044
ASTORGA,0.245551,0.246899,0.211425,0.194686,0.215953,0.211714,0.232863,0.18068,0.15165,0.188187
BARRACÃO,0.173661,0.138861,0.210286,0.19232,0.16164,0.129046,0.120696,0.185361,0.16917,0.126557
CAMPO LARGO,0.267975,0.404535,0.293833,0.317331,0.275311,0.296047,0.491012,0.332771,0.343311,0.319032
CAPANEMA,0.352819,0.539397,0.409939,0.430239,0.364272,0.351762,0.676248,0.461265,0.496933,0.402251
CASCAVEL,0.26059,0.332521,0.260665,0.274501,0.261245,0.245421,0.371156,0.296631,0.295241,0.273319
CORONEL VIVIDA,0.278768,0.336191,0.299007,0.269459,0.271586,0.246694,0.391226,0.259948,0.238781,0.245954
CURITIBA,0.259464,0.394579,0.299062,0.320178,0.264983,0.22737,0.412633,0.30465,0.249723,0.235209
FOZ DO IGUAÇU,0.404487,0.691073,0.43975,0.409392,0.408254,0.415203,0.805284,0.4235,0.359564,0.405657
GOIOERÊ,0.266711,0.35742,0.285289,0.270382,0.272007,0.250127,0.423674,0.30849,0.277902,0.272567
